# TalkTalk NBA — App Replica (Python)

A pure-Python reproduction of every chart, statistic and customer-level output the live decisioning app produces. Each section is mapped to a screen in the app so a reader can compare side-by-side.

| Section | App screen |
|--|--|
| 1 — ROI & Exec Summary | `/` |
| 2 — Strategy & Pipeline | `/strategy` |
| 3 — Model Evaluation | `/model` |
| 4 — Explainability | `/explainability` |
| 5 — NBA Rules | `/nba-rules` |
| 6 — Customer Lookup | Explainability drawer |
| 7 — Top-50 most impacted | Explainability table |

**Inputs (all in this folder):** `model_metrics.json`, `feature_importance.csv`, `nba_roi_params.json`, `top_50_customers.json`, `segment_risk_summary.csv`, `lovable_sample_customer_info.csv`, `lovable_sample_calls.csv`, `lovable_sample_cease.csv`, `lovable_sample_usage.csv`.

Run top-to-bottom. The customer-lookup cell (section 6) is parameterised — edit `CUSTOMER_ID` and re-run.

## 0 · Setup & data loading

In [ ]:
from __future__ import annotations
import json, math
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

DATA = Path('.')
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 10,
})

# Midnight Executive palette — same as the in-app charts.
NAVY, ICE, CORAL, AMBER, MINT, SLATE = '#0B1220', '#7DD3FC', '#F87171', '#F59E0B', '#34D399', '#94A3B8'
TIER_COLOR = {'High': CORAL, 'Medium': AMBER, 'Low': MINT}

def fmt_gbp(v, compact=False):
    s = '-' if v < 0 else ''
    a = abs(v)
    if compact:
        if a >= 1e9: return f'{s}£{round(a/1e9)}B'
        if a >= 1e6: return f'{s}£{round(a/1e6)}M'
        if a >= 1e3: return f'{s}£{round(a/1e3)}K'
        return f'{s}£{round(a)}'
    return f'{s}£{round(a):,}'

def fmt_int(v): return f'{round(v):,}'
def fmt_pct(v, dp=1): return f'{v*100:.{dp}f}%'

metrics      = json.loads((DATA/'model_metrics.json').read_text())
roi_params   = json.loads((DATA/'nba_roi_params.json').read_text())
top50        = json.loads((DATA/'top_50_customers.json').read_text())
feat_imp     = pd.read_csv(DATA/'feature_importance.csv')
segment_csv  = pd.read_csv(DATA/'segment_risk_summary.csv')
cust_info    = pd.read_csv(DATA/'lovable_sample_customer_info.csv')
calls        = pd.read_csv(DATA/'lovable_sample_calls.csv')
cease        = pd.read_csv(DATA/'lovable_sample_cease.csv')
try:
    usage = pd.read_csv(DATA/'lovable_sample_usage.csv')
except FileNotFoundError:
    usage = pd.DataFrame(columns=['unique_customer_identifier','calendar_date','usage_download_mbs','usage_upload_mbs'])

for c in ('hold_time_seconds','talk_time_seconds'):
    if c in calls: calls[c] = pd.to_numeric(calls[c], errors='coerce')
for c in ('usage_download_mbs','usage_upload_mbs'):
    if c in usage: usage[c] = pd.to_numeric(usage[c], errors='coerce')

print(f'Loaded {len(cust_info):,} customers, {len(calls):,} call rows, {len(usage):,} usage rows, {len(cease):,} cease rows.')
print(f'Model: {metrics["model_type"]} · ROC-AUC {metrics["performance_metrics"]["roc_auc"]:.3f} · threshold {metrics["performance_metrics"]["decision_threshold"]:.2f}')

### NBA rules, triggers and financial helpers (mirrors `src/data/*`)

These constants are copied 1:1 from `src/data/customers.ts`, `src/data/financials.ts` and the live `nba_rules` table so the Python output matches the app at the same scenario inputs.

In [ ]:
NBA_TRIGGERS = {
    'loyalty_save_desk': dict(label='Specialist Save Desk', channel='Outbound Call',
        offer='20% loyalty discount + 24-month re-contract'),
    'free_tech_upgrade': dict(label='Free Tech Upgrade', channel='Outbound Call + Engineer Visit',
        offer='FTTC → G.Fast / FTTP migration, no install fee'),
    'rightsize_email': dict(label='Right-size Upgrade Email', channel='Email + In-app',
        offer='Premium fibre package, 6-month price hold'),
    'competitor_match': dict(label='Competitor-match Save Offer', channel="Customer's preferred channel",
        offer='Top-tier price match (£/month off)'),
    'suppress':          dict(label='Do Not Disturb', channel='Suppress', offer='Annual thank-you only'),
    'nurture':           dict(label='Personalised Nurture', channel='Email', offer='Account review + bill explainer'),
}

DEFAULT_RULES = pd.DataFrame([
    dict(trigger_key='loyalty_save_desk', label='Specialist Save Desk',        channel='Outbound Call',                  discount_pct=20, contract_months=24, cost_per_contact_gbp=12.0, is_active=True),
    dict(trigger_key='free_tech_upgrade', label='Free Tech Upgrade',           channel='Outbound Call + Engineer Visit', discount_pct=0,  contract_months=24, cost_per_contact_gbp=65.0, is_active=True),
    dict(trigger_key='rightsize_email',   label='Right-size Upgrade Email',    channel='Email + In-app',                 discount_pct=10, contract_months=18, cost_per_contact_gbp=0.4,  is_active=True),
    dict(trigger_key='competitor_match',  label='Competitor-match Save Offer', channel="Customer's preferred channel",   discount_pct=25, contract_months=24, cost_per_contact_gbp=14.0, is_active=True),
    dict(trigger_key='suppress',          label='Do Not Disturb',              channel='Suppress',                       discount_pct=0,  contract_months=0,  cost_per_contact_gbp=0.0,  is_active=True),
    dict(trigger_key='nurture',           label='Personalised Nurture',        channel='Email',                          discount_pct=5,  contract_months=12, cost_per_contact_gbp=0.3,  is_active=True),
])

RULE_MIX = {'loyalty_save_desk': 0.32, 'free_tech_upgrade': 0.18, 'rightsize_email': 0.22,
            'competitor_match': 0.10, 'nurture': 0.18, 'suppress': 0.0}
TIER_ANNUAL_CHURN = {'High': 0.45, 'Medium': 0.18, 'Low': 0.06}

def tier_horizon_months(tier): return min(120, 12 / TIER_ANNUAL_CHURN[tier])
def customer_ltv(monthly_arpu, tier): return monthly_arpu * tier_horizon_months(tier)

def compute_rule_financials(rules: pd.DataFrame, high_risk_volume, monthly_arpu, success_rate, primary_tier='High'):
    active = rules[(rules.is_active) & (rules.trigger_key != 'suppress')].copy()
    active['weight'] = active['trigger_key'].map(RULE_MIX).fillna(0.0)
    total = active['weight'].sum()
    if total == 0: return pd.DataFrame()
    active['share'] = active['weight'] / total
    rows = []
    for _, r in active.iterrows():
        contacted = round(high_risk_volume * r.share)
        saved = round(contacted * success_rate)
        horizon = r.contract_months if r.contract_months > 0 else tier_horizon_months(primary_tier)
        gross = saved * monthly_arpu * horizon
        dilution = gross * (r.discount_pct / 100)
        cost = contacted * r.cost_per_contact_gbp
        rows.append(dict(trigger_key=r.trigger_key, label=r.label, channel=r.channel,
                         contacted=contacted, saved=saved,
                         gross_retained_gbp=round(gross), dilution_gbp=round(dilution),
                         cost_to_serve_gbp=round(cost), net_retained_gbp=round(gross - dilution - cost)))
    return pd.DataFrame(rows)

print('Helpers ready.')

### `score_customer` — Python mirror of `src/data/scoring.ts`

Identical base score (0.5), feature impact weights, risk-tier thresholds and NBA-trigger routing rules. This is what powers section 6 (customer drawer).

In [ ]:
def tier_from_score(s):
    if s >= 0.65: return 'High'
    if s >= 0.35: return 'Medium'
    return 'Low'

def normalise_contract(raw):
    s = (raw or '').lower()
    if 'ooc' in s or 'out of contract' in s: return 'Out of contract'
    if 'rolling' in s: return 'Rolling'
    return 'In contract'

def derive_nba_trigger(risk_tier, contract_status, signals, package):
    speed_deficit = (signals['sold_speed_mbps'] - signals['line_speed_mbps']) / signals['sold_speed_mbps'] if signals['sold_speed_mbps'] else 0
    heavy = signals.get('monthly_download_gb', 0) > 800 and any(p in (package or '') for p in ('Fibre 35','Fibre 65','ADSL','Essentials'))
    if risk_tier == 'Low': return 'suppress'
    if signals.get('cease_insight') == 'CompetitorDeals': return 'competitor_match'
    if signals.get('loyalty_calls_90d', 0) >= 2 or signals.get('total_hold_seconds', 0) > 1800: return 'loyalty_save_desk'
    if speed_deficit > 0.25 or any(p in (package or '') for p in ('ADSL','Fibre 35')): return 'free_tech_upgrade'
    if heavy: return 'rightsize_email'
    if risk_tier == 'High' and contract_status == 'Out of contract': return 'loyalty_save_desk'
    return 'nurture'

def score_customer(inp: dict) -> dict:
    base = 0.5
    c = []
    ooc = max(-0.05, min(0.30, (inp['ooc_days']/600)*0.30))
    c.append(dict(feature='ooc_days', label='Days Out of Contract', impact=round(ooc,3),
        detail=f"{inp['ooc_days']} days since contract end." if inp['ooc_days']>0 else f"{abs(inp['ooc_days'])} days remaining on contract."))
    dd = 0.18 if inp['dd_cancel_60'] > 0 else -0.02
    c.append(dict(feature='dd_cancel_60_day', label='Recent DD Cancel (60d)', impact=round(dd,3),
        detail='Direct Debit cancelled in the last 60 days.' if inp['dd_cancel_60']>0 else 'No recent DD failures.'))
    ddl = min(0.12, inp['contract_dd_cancels']*0.04)
    c.append(dict(feature='contract_dd_cancels', label='DD Cancellations', impact=round(ddl,3),
        detail=f"{inp['contract_dd_cancels']} DD cancellation(s) in account history."))
    ten = -min(0.32, (inp['tenure_days']/4000)*0.32)
    c.append(dict(feature='tenure_days', label='Customer Tenure', impact=round(ten,3),
        detail=f"{inp['tenure_days']/365:.1f} years of tenure."))
    sd_pct = 0.0
    if inp['sold_speed_mbps'] > 0 and inp['line_speed_mbps'] >= 0:
        sd_pct = (inp['sold_speed_mbps'] - inp['line_speed_mbps'])/inp['sold_speed_mbps']
        if sd_pct > 0.1:
            sd = min(0.16, sd_pct*0.2)
            c.append(dict(feature='speed_deficit', label='Speed Deficit', impact=round(sd,3),
                detail=f"Receiving {inp['line_speed_mbps']:.1f} Mbps vs {inp['sold_speed_mbps']} Mbps sold ({sd_pct*100:.0f}% deficit)."))
    loy = inp.get('loyalty_calls_90d', 0)
    if loy > 0:
        c.append(dict(feature='loyalty_calls', label='Loyalty Calls', impact=round(min(0.22, loy*0.07),3),
            detail=f"{loy} loyalty call(s) in last 90 days."))
    hold = inp.get('total_hold_seconds', 0)
    if hold > 600:
        c.append(dict(feature='total_hold_time', label='Total Hold Time', impact=round(min(0.12,(hold/3600)*0.08),3),
            detail=f"{round(hold/60)} minutes on hold across recent calls."))
    if inp.get('cease_insight') == 'CompetitorDeals':
        c.append(dict(feature='cease_competitor', label='Cease Pattern · Competitor', impact=0.15,
            detail='Profile matches historical Competitor Deals cease patterns.'))
    dl = inp.get('monthly_download_gb', 0)
    if dl > 800 and any(p in (inp['package'] or '') for p in ('Fibre 35','Fibre 65','ADSL','Essentials')):
        c.append(dict(feature='usage_overflow', label='Usage vs Package', impact=0.08,
            detail=f'{round(dl)} GB/mo on a basic package — capacity-bound.'))
    score = max(0.02, min(0.98, base + sum(x['impact'] for x in c)))
    c.sort(key=lambda x: abs(x['impact']), reverse=True)
    tier = tier_from_score(score)
    contract = normalise_contract(inp['contract_status_raw'])
    signals = dict(loyalty_calls_90d=loy, total_hold_seconds=hold,
                   total_talk_seconds=inp.get('total_talk_seconds',0), ooc_days=inp['ooc_days'],
                   sold_speed_mbps=inp['sold_speed_mbps'], line_speed_mbps=inp['line_speed_mbps'],
                   monthly_download_gb=dl, monthly_upload_gb=inp.get('monthly_upload_gb',0),
                   cease_insight=inp.get('cease_insight'), technology=inp.get('technology',''))
    trig_key = derive_nba_trigger(tier, contract, signals, inp['package'])
    trig = NBA_TRIGGERS[trig_key]
    top = c[:3]
    drivers = ', '.join(f"{x['label'].lower()} ({'+' if x['impact']>=0 else ''}{x['impact']*100:.0f} pts)" for x in top) or 'no dominant signal'
    why_cust = (f"{tier} risk (score {score*100:.0f}/100). Strongest drivers: {drivers}. " +
        (f"Currently out of contract for {inp['ooc_days']} days — free to switch on any given day." if contract=='Out of contract' else
         'On a rolling monthly contract — low switching friction.' if contract=='Rolling' else
         f"In contract with {abs(inp['ooc_days'])} days left."))
    why_nba = {
        'loyalty_save_desk': ('Multiple loyalty calls / extended hold time signal active shopping — needs a specialist save agent with a pre-approved discount before they cancel.'
            if loy>=2 or hold>1800 else
            'High-risk customer out of contract — proactive call with a loyalty discount converts at +18 ppt vs. control.'),
        'free_tech_upgrade': f"Sold {inp['sold_speed_mbps']} Mbps but receiving {inp['line_speed_mbps']} Mbps ({sd_pct*100:.0f}% deficit). Discounting masks the real problem — fix the line first.",
        'rightsize_email':   f"Heavy usage ({round(dl)} GB/mo) on a basic package — they will keep hitting throttling. An automated upgrade email converts at +9 ppt.",
        'competitor_match':  'Cease intent matches historical Competitor Deals patterns — price is the primary lever, route the highest-tier price-match offer through their preferred channel.',
        'suppress':          'Long-tenure low-risk customer. Outbound contact erodes satisfaction here — hold in nurture sequences only.',
        'nurture':           'Mid-risk profile without a single dominant trigger. Send a personalised retention email with usage insights to keep the relationship warm.',
    }[trig_key]
    return dict(customer=dict(id=inp['id'], name=inp.get('name', f"Customer {inp['id'][:6]}"),
                              package=inp['package'], tenure_days=inp['tenure_days'], contract_status=contract),
                risk_score=round(score,3), risk_tier=tier, base_score=base, shap=c,
                nba=dict(trigger=trig_key, **trig), signals=signals,
                why_this_customer=why_cust, why_this_nba=why_nba,
                speed_deficit_pct=round(sd_pct,3))

print('score_customer ready.')

## 1 · ROI & Exec Summary  *(mirrors `/`)*

In [ ]:
DEFAULT_SUCCESS = 0.18
incremental_saved = roi_params['high_risk_volume'] * (DEFAULT_SUCCESS - roi_params['baseline_retention_conversion_rate'])
projected_saved_revenue = incremental_saved * roi_params['average_annual_arpu_gbp']
monthly_arpu = roi_params['average_annual_arpu_gbp'] / 12

kpis = pd.DataFrame([
    ('Total customer base',      fmt_int(roi_params['total_customer_base'])),
    ('High-risk volume',         fmt_int(roi_params['high_risk_volume'])),
    ('Average annual ARPU',      fmt_gbp(roi_params['average_annual_arpu_gbp'])),
    ('Baseline conversion',      fmt_pct(roi_params['baseline_retention_conversion_rate'])),
    ('Revenue at risk',          fmt_gbp(roi_params['revenue_at_risk_gbp'], compact=True)),
    (f'Projected saved revenue @ {int(DEFAULT_SUCCESS*100)}%', fmt_gbp(projected_saved_revenue, compact=True)),
], columns=['Metric','Value']).set_index('Metric')
kpis

In [ ]:
# Risk-tier mix pie
tier_map = {'High Risk':'High','Medium Risk':'Medium','Low Risk':'Low'}
seg = segment_csv.assign(tier=segment_csv['Risk_Tier'].map(tier_map))
fig, ax = plt.subplots(figsize=(5.5,5.5))
ax.pie(seg['Customer_Count'], labels=[f"{t}\n{fmt_int(c)}" for t,c in zip(seg.tier, seg.Customer_Count)],
       colors=[TIER_COLOR[t] for t in seg.tier], wedgeprops=dict(width=0.42, edgecolor='white'),
       startangle=90, textprops={'fontsize':10})
ax.set_title('Risk-tier mix · 3.5M customer base'); plt.show()

In [ ]:
# Net ROI per NBA trigger (RoiSimulator + PerTriggerSensitivityPanel)
rule_fin = compute_rule_financials(DEFAULT_RULES, roi_params['high_risk_volume'], monthly_arpu, DEFAULT_SUCCESS)
fig, ax = plt.subplots(figsize=(9,4.2))
colors = [MINT if v>=0 else CORAL for v in rule_fin['net_retained_gbp']]
ax.barh(rule_fin['label'], rule_fin['net_retained_gbp']/1e6, color=colors, edgecolor='white')
ax.set_xlabel('Net retained revenue (£M)')
ax.set_title(f'Net ROI per NBA trigger · success rate {int(DEFAULT_SUCCESS*100)}%')
for i,v in enumerate(rule_fin['net_retained_gbp']/1e6):
    ax.text(v, i, f'  £{v:,.0f}M', va='center', fontsize=9)
ax.invert_yaxis(); plt.tight_layout(); plt.show()

totals = rule_fin[['contacted','saved','gross_retained_gbp','dilution_gbp','cost_to_serve_gbp','net_retained_gbp']].sum()
print(f"Portfolio · contacted {fmt_int(totals.contacted)} · saved {fmt_int(totals.saved)} · "
      f"net {fmt_gbp(totals.net_retained_gbp, compact=True)} (gross {fmt_gbp(totals.gross_retained_gbp, compact=True)} − "
      f"dilution {fmt_gbp(totals.dilution_gbp, compact=True)} − cost {fmt_gbp(totals.cost_to_serve_gbp, compact=True)})")

In [ ]:
# Sensitivity: Net ROI vs success rate
rates = np.arange(0.10, 0.36, 0.01)
nets = [compute_rule_financials(DEFAULT_RULES, roi_params['high_risk_volume'], monthly_arpu, r)['net_retained_gbp'].sum() for r in rates]
fig, ax = plt.subplots(figsize=(9,3.6))
ax.plot(rates*100, np.array(nets)/1e6, color=ICE, lw=2.4, marker='o', ms=4)
ax.axvline(DEFAULT_SUCCESS*100, color=CORAL, ls='--', lw=1, label=f'Default {int(DEFAULT_SUCCESS*100)}%')
ax.set_xlabel('Save-call success rate (%)'); ax.set_ylabel('Portfolio net ROI (£M)')
ax.set_title('Sensitivity · Net ROI vs success rate'); ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Net ROI segment drilldown (NetRoiSegmentDrilldown):
# split portfolio net ROI by contract status and tenure bucket using volumes
# from the live customer sample as a realistic distribution.
share = (cust_info
         .assign(contract=cust_info['contract_status'].apply(normalise_contract),
                 tenure_bucket=pd.cut(cust_info['tenure_days'],
                                      bins=[-1,365,730,1460,3650,99999],
                                      labels=['0-1y','1-2y','2-4y','4-10y','10y+']))
         .groupby(['tenure_bucket','contract'], observed=True).size().unstack(fill_value=0))
share = share / share.values.sum()
portfolio_net = float(rule_fin['net_retained_gbp'].sum())
drill = (share * portfolio_net / 1e6).round(2)
ax = drill.plot(kind='bar', stacked=True, figsize=(9,4),
                color=[CORAL, AMBER, ICE][:drill.shape[1]], edgecolor='white')
ax.set_title('Net ROI drilldown by tenure × contract status (£M)')
ax.set_ylabel('Net ROI (£M)'); ax.set_xlabel('Tenure bucket'); ax.legend(title='Contract')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()
drill

## 2 · Strategy & Pipeline  *(mirrors `/strategy`)*

In [ ]:
pipeline = '''
Databricks / DuckDB         scikit-learn / XGBoost      NBA Rule Engine            Contact Centre
+--------------------+      +---------------------+     +----------------------+    +----------------+
|  Data Ingestion    | ───▶ |  Predictive Model   | ──▶ |  Risk × Context →    | ─▶ |  Save calls /  |
|  customer_info,    |      |  AUC 0.87, weekly   |     |  Trigger + Channel   |    |  emails / SMS  |
|  calls, usage      |      |  retrain, SHAP attr.|     |  + offer + cost      |    |                |
+--------------------+      +---------------------+     +----------------------+    +----------------+
'''
print(pipeline)

treatment_matrix = pd.DataFrame([
    ('High Risk · Out of Contract', 'High',   'OOC > 60d, Fibre, talk time elevated', 'loyalty_save_desk', 'Outbound Call',     '+18 ppt', 0.21),
    ('High Risk · In Contract',     'High',   'Speed deficit > 30%, recent DD issues', 'free_tech_upgrade','Outbound Call + SMS','+12 ppt', 0.08),
    ('Medium Risk · High Usage',    'Medium', 'Fibre 65 hitting bandwidth ceiling',    'rightsize_email',  'Email + In-app',    '+9 ppt',  0.19),
    ('Medium Risk · Loyalty Caller','Medium', '≥1 loyalty call in last 90d, in contract','nurture',        'Email',             '+6 ppt',  0.16),
    ('Low Risk · Long Tenure',      'Low',    'Tenure > 8 yrs, no recent service events','suppress',       'Suppress',          'Avoid £4.20', 0.28),
    ('Low Risk · New Customer',     'Low',    'Tenure < 180 days, healthy usage',     'nurture',          'Email + In-app',    'Brand affinity', 0.08),
], columns=['segment','risk_tier','context','trigger','channel','expected_lift','share_of_base'])
treatment_matrix

## 3 · Model Evaluation  *(mirrors `/model`)*

In [ ]:
perf = metrics['performance_metrics']
split = metrics['dataset_split']
hp = metrics['hyperparameters']

print(f"Model: {metrics['model_type']}  ·  trained {metrics['trained_at']}")
print(f"Train/Test: {fmt_int(split['train_size'])} / {fmt_int(split['test_size'])} rows")
key_hp = {k: hp[k] for k in ['objective','learning_rate','max_depth','n_estimators',
                              'subsample','colsample_bytree','tree_method','eval_metric','random_state'] if k in hp}
pd.Series(key_hp, name='value').to_frame()

In [ ]:
# Performance bar
p = perf
fig, ax = plt.subplots(figsize=(8,3.4))
labels = ['Accuracy','Precision','Recall','F1','ROC-AUC']
vals = [p['accuracy'], p['precision'], p['recall'], p['f1_score'], p['roc_auc']]
bars = ax.bar(labels, vals, color=[ICE,ICE,ICE,ICE,CORAL], edgecolor='white')
for b,v in zip(bars,vals): ax.text(b.get_x()+b.get_width()/2, v+0.01, f'{v:.3f}', ha='center', fontsize=9)
ax.set_ylim(0,1); ax.set_title(f"Performance metrics · decision threshold {p['decision_threshold']:.2f}")
plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrix heatmap
cm = metrics['confusion_matrix']
M = np.array([[cm['true_negatives'], cm['false_positives']],
              [cm['false_negatives'], cm['true_positives']]])
fig, ax = plt.subplots(figsize=(5.5,4.2))
im = ax.imshow(M, cmap='Blues')
for i in range(2):
    for j in range(2):
        pct = M[i,j] / M[i].sum() * 100
        ax.text(j, i, f"{fmt_int(M[i,j])}\n({pct:.1f}%)", ha='center', va='center',
                color='white' if M[i,j] > M.max()/2 else NAVY, fontsize=10)
ax.set_xticks([0,1], ['Pred Stay','Pred Churn'])
ax.set_yticks([0,1], ['Actual Stay','Actual Churn'])
ax.set_title('Confusion matrix · row %'); plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

In [ ]:
# ROC curve
roc = pd.DataFrame(metrics['roc_curve']).replace([np.inf,-np.inf], np.nan).dropna(subset=['fpr','tpr'])
op = roc.iloc[(roc['threshold'] - perf['decision_threshold']).abs().argsort()[:1]].iloc[0]
fig, ax = plt.subplots(figsize=(5.6,5))
ax.plot(roc.fpr, roc.tpr, color=ICE, lw=2.2, label=f"AUC {perf['roc_auc']:.3f}")
ax.plot([0,1],[0,1], color=SLATE, ls='--', lw=1)
ax.scatter([op.fpr],[op.tpr], color=CORAL, zorder=5, label=f"Threshold {perf['decision_threshold']:.2f}")
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
ax.set_title('ROC curve'); ax.legend(loc='lower right'); plt.tight_layout(); plt.show()

In [ ]:
# Per-segment precision / recall
seg_m = pd.DataFrame(metrics['segment_metrics'])
x = np.arange(len(seg_m)); w = 0.38
fig, ax = plt.subplots(figsize=(8,3.6))
ax.bar(x-w/2, seg_m['precision'], w, color=ICE, label='Precision', edgecolor='white')
ax.bar(x+w/2, seg_m['recall'],    w, color=CORAL, label='Recall',    edgecolor='white')
ax.set_xticks(x, [f"{s}\n(n={fmt_int(n)})" for s,n in zip(seg_m['segment'], seg_m['n'])])
ax.set_ylim(0,1.05); ax.set_title('Per-tenure-segment performance'); ax.legend()
plt.tight_layout(); plt.show()

## 4 · Explainability  *(mirrors `/explainability`)*

In [ ]:
fi = feat_imp.sort_values('importance', ascending=True)
fig, ax = plt.subplots(figsize=(8.5, 0.32*len(fi)+1.5))
ax.barh(fi['feature'], fi['importance'], color=ICE, edgecolor='white')
for y,(f,v) in enumerate(zip(fi['feature'], fi['importance'])):
    ax.text(v, y, f'  {v:.3f}', va='center', fontsize=9)
ax.set_title('Global feature importance · XGBoost gain'); plt.tight_layout(); plt.show()

In [ ]:
# Score the local sample so we can colour distributions by predicted risk tier.
def row_to_input(r):
    return dict(id=r['unique_customer_identifier'], package=r.get('crm_package_name','') or '',
                tenure_days=int(r['tenure_days']), contract_status_raw=r.get('contract_status','') or '',
                ooc_days=int(r['ooc_days']), dd_cancel_60=int(r['dd_cancel_60_day']),
                contract_dd_cancels=int(r['contract_dd_cancels']),
                sold_speed_mbps=float(r['speed']) if pd.notna(r['speed']) else 0.0,
                line_speed_mbps=float(r['line_speed']) if pd.notna(r['line_speed']) else 0.0,
                technology=r.get('technology',''))

scored = cust_info.copy()
results = scored.apply(lambda r: score_customer(row_to_input(r)), axis=1)
scored['risk_score'] = results.apply(lambda x: x['risk_score'])
scored['risk_tier']  = results.apply(lambda x: x['risk_tier'])

fig, axes = plt.subplots(1,3, figsize=(13,3.6))
for ax, col, title in zip(axes,
    ['tenure_days','ooc_days','contract_dd_cancels'],
    ['Tenure (days)','Days out of contract','DD cancellations']):
    for tier in ['Low','Medium','High']:
        sub = scored[scored.risk_tier==tier][col].dropna()
        if len(sub):
            ax.hist(sub, bins=25, color=TIER_COLOR[tier], alpha=0.65, label=tier, edgecolor='white')
    ax.set_title(title); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
scored['risk_tier'].value_counts().reindex(['High','Medium','Low']).rename('count').to_frame()

In [ ]:
# Local explanation for a top-50 customer (mirrors the in-app drawer SHAP plot).
demo = top50['customers'][0]
rs = pd.DataFrame(demo['reason_codes'])
fig, ax = plt.subplots(figsize=(8,3.2))
colors = [CORAL if v>=0 else MINT for v in rs['impact']]
ax.barh(rs['feature'], rs['impact'], color=colors, edgecolor='white')
ax.axvline(0, color=SLATE, lw=0.8)
ax.set_title(f"Local explanation · {demo['customer_id'][:10]}…  churn prob {demo['churn_prob']:.3f}")
ax.set_xlabel('SHAP-style impact')
ax.invert_yaxis(); plt.tight_layout(); plt.show()
print('Recommended NBA :', demo['recommended_nba'])
print('Expected save   :', fmt_gbp(demo['expected_save_gbp']))

## 5 · NBA Rules  *(mirrors `/nba-rules`)*

In [ ]:
rules_view = DEFAULT_RULES[['trigger_key','label','channel','discount_pct','contract_months','cost_per_contact_gbp','is_active']]
display(rules_view)

# Expected save GBP per trigger at default success rate
fig, ax = plt.subplots(figsize=(8.5,3.6))
ax.bar(rule_fin['label'], rule_fin['gross_retained_gbp']/1e6, color=ICE, label='Gross retained', edgecolor='white')
ax.bar(rule_fin['label'], -rule_fin['dilution_gbp']/1e6,      color=AMBER, label='Discount dilution', edgecolor='white')
ax.bar(rule_fin['label'], -rule_fin['cost_to_serve_gbp']/1e6, color=CORAL, label='Cost to serve',
       bottom=-rule_fin['dilution_gbp']/1e6, edgecolor='white')
ax.set_ylabel('£M'); ax.set_title('Per-trigger economics · gross vs cost components')
ax.legend(); plt.xticks(rotation=18, ha='right'); plt.tight_layout(); plt.show()

## 6 · Customer Lookup  *(the in-app drawer, in Python)*

Edit `CUSTOMER_ID` and re-run the cell. The function searches the local sample (`lovable_sample_customer_info.csv`) and joins recent calls / usage / cease records, then runs the Python `score_customer` to produce the same risk score, SHAP contributions, NBA trigger and narrative the in-app drawer renders.

If the id is not in the sample, the cell falls back to `top_50_customers.json` so the format is still demonstrated end-to-end.

In [ ]:
def lookup_customer(cid: str) -> Optional[dict]:
    row = cust_info[cust_info['unique_customer_identifier'] == cid]
    if len(row) == 0:
        # fall back to top-50 export
        for c in top50['customers']:
            if c['customer_id'] == cid or cid == 'top1':
                f = c['features']
                inp = dict(id=c['customer_id'], package=str(f.get('crm_package_name') or ''),
                           tenure_days=int(f.get('tenure_days') or 0),
                           contract_status_raw=str(f.get('contract_status') or ''),
                           ooc_days=int(f.get('ooc_days') or 0),
                           dd_cancel_60=int(f.get('dd_cancel_60_day') or 0),
                           contract_dd_cancels=int(f.get('contract_dd_cancels') or 0),
                           sold_speed_mbps=float(f.get('speed') or 0),
                           line_speed_mbps=float(f.get('line_speed') or 0),
                           technology=str(f.get('technology') or ''),
                           loyalty_calls_90d=int(f.get('loyalty_calls_90d') or 0),
                           total_hold_seconds=float(f.get('avg_hold_seconds') or 0),
                           total_talk_seconds=float(f.get('avg_talk_seconds') or 0),
                           monthly_download_gb=float(f.get('avg_download_mbs') or 0),
                           monthly_upload_gb=float(f.get('avg_upload_mbs') or 0))
                profile = score_customer(inp)
                profile['source'] = 'top_50_customers.json'
                profile['model_churn_prob'] = c['churn_prob']
                profile['expected_save_gbp'] = c['expected_save_gbp']
                return profile
        return None
    r = row.iloc[0].to_dict()
    cust_calls = calls[calls['unique_customer_identifier']==cid]
    cust_usage = usage[usage['unique_customer_identifier']==cid]
    cust_cease = cease[cease['unique_customer_identifier']==cid]
    inp = row_to_input(r)
    inp['loyalty_calls_90d']  = int((cust_calls['call_type']=='Loyalty').sum())
    inp['total_hold_seconds'] = float(cust_calls['hold_time_seconds'].fillna(0).sum())
    inp['total_talk_seconds'] = float(cust_calls['talk_time_seconds'].fillna(0).sum())
    if len(cust_usage):
        # treat usage_*_mbs as monthly GB proxy
        inp['monthly_download_gb'] = float(cust_usage['usage_download_mbs'].fillna(0).mean())
        inp['monthly_upload_gb']   = float(cust_usage['usage_upload_mbs'].fillna(0).mean())
    if len(cust_cease):
        inp['cease_insight'] = str(cust_cease.iloc[0]['reason_description_insight'])
    profile = score_customer(inp)
    profile['source'] = 'lovable_sample_customer_info.csv'
    profile['recent_calls'] = cust_calls.copy()
    profile['recent_usage'] = cust_usage.copy()
    return profile

def render_customer_profile(p: dict):
    if p is None:
        print('Customer not found.'); return
    c, n, s = p['customer'], p['nba'], p['signals']
    print('━'*78)
    print(f"Customer {c['id']}  ·  source: {p.get('source','-')}")
    print('━'*78)
    print(f"Package        : {c['package']}")
    print(f"Tenure         : {c['tenure_days']/365:.1f} years ({c['tenure_days']} days)")
    print(f"Contract       : {c['contract_status']}")
    print()
    print(f"Risk score     : {p['risk_score']*100:.0f}/100   tier: {p['risk_tier']}")
    if 'model_churn_prob' in p:
        print(f"Model churn p  : {p['model_churn_prob']:.3f}")
        print(f"Expected save  : {fmt_gbp(p['expected_save_gbp'])}")
    print(f"Recommended    : {n['label']}  ({n['trigger']})")
    print(f"Channel        : {n['channel']}")
    print(f"Offer          : {n['offer']}")
    print()
    print('Behavioural signals')
    sig_tbl = pd.Series({
        'Loyalty calls (90d)': s['loyalty_calls_90d'],
        'Total hold (s)':      round(s['total_hold_seconds']),
        'Total talk (s)':      round(s['total_talk_seconds']),
        'OOC days':            s['ooc_days'],
        'Sold speed (Mbps)':   s['sold_speed_mbps'],
        'Line speed (Mbps)':   round(s['line_speed_mbps'],2),
        'Speed deficit':       fmt_pct(p['speed_deficit_pct'],0),
        'Monthly download':    round(s['monthly_download_gb'],1),
        'Cease insight':       s.get('cease_insight') or '—',
    }, name='value').to_frame()
    display(sig_tbl)
    print('Why this customer')
    print(' ', p['why_this_customer'])
    print('\nWhy this NBA')
    print(' ', p['why_this_nba'])
    # SHAP bar chart
    shap_df = pd.DataFrame(p['shap']).head(6)
    fig, ax = plt.subplots(figsize=(8.5, 0.45*len(shap_df)+1.2))
    colors = [CORAL if v>=0 else MINT for v in shap_df['impact']]
    ax.barh(shap_df['label'], shap_df['impact'], color=colors, edgecolor='white')
    for y,(v,d) in enumerate(zip(shap_df['impact'], shap_df['detail'])):
        ax.text(v, y, f"  {'+' if v>=0 else ''}{v*100:.0f} pts", va='center', fontsize=9)
    ax.axvline(0, color=SLATE, lw=0.8); ax.invert_yaxis()
    ax.set_title(f"Local explanation · risk score {p['risk_score']*100:.0f}/100 ({p['risk_tier']})")
    ax.set_xlabel('Impact on risk score (points)')
    plt.tight_layout(); plt.show()
    # Recent calls / usage timeline if available
    rc = p.get('recent_calls')
    if rc is not None and len(rc):
        rc = rc.copy()
        rc['event_date'] = pd.to_datetime(rc['event_date'], errors='coerce')
        fig, ax = plt.subplots(figsize=(8.5,2.6))
        ax.scatter(rc['event_date'], rc['talk_time_seconds'].fillna(0), s=60, color=ICE, label='Talk (s)')
        ax.scatter(rc['event_date'], rc['hold_time_seconds'].fillna(0), s=40, color=CORAL, label='Hold (s)', marker='x')
        ax.set_title('Recent contact-centre interactions'); ax.legend()
        plt.tight_layout(); plt.show()
    print('Detail (impact ranked):')
    for x in p['shap']:
        print(f"  {x['label']:<32} {('+' if x['impact']>=0 else '')+f'{x["impact"]*100:5.0f}'} pts  · {x['detail']}")

In [ ]:
# ── Edit and re-run ────────────────────────────────────────────────────────
CUSTOMER_ID = cust_info['unique_customer_identifier'].iloc[0]
# Examples to try:
#   - any value from cust_info['unique_customer_identifier']
#   - any customer_id from top_50_customers.json
#   - 'top1' (resolves to the single highest-risk customer in top50)
render_customer_profile(lookup_customer(CUSTOMER_ID))

## 7 · Top-50 most impacted  *(mirrors the live table)*

In [ ]:
t50 = pd.DataFrame([{
    'rank': c['rank'], 'customer_id': c['customer_id'],
    'churn_prob': c['churn_prob'],
    'recommended_nba': c['recommended_nba'],
    'expected_save_gbp': c['expected_save_gbp'],
} for c in top50['customers']])
display(t50.head(20))
print(f"Threshold {top50['threshold']:.2f}  ·  scored {top50['scored_n']:,} customers  ·  showing top {len(t50)}")

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(13,3.8))
by_nba = t50.groupby('recommended_nba')['expected_save_gbp'].sum().sort_values()
axes[0].barh(by_nba.index, by_nba.values, color=ICE, edgecolor='white')
axes[0].set_title('Top-50 · expected save by NBA (£)')
for y,v in enumerate(by_nba.values):
    axes[0].text(v, y, f'  £{v:,.0f}', va='center', fontsize=8)
axes[1].hist(t50['churn_prob'], bins=20, color=CORAL, edgecolor='white')
axes[1].set_title('Top-50 · churn probability distribution'); axes[1].set_xlabel('churn_prob')
plt.tight_layout(); plt.show()

---
**Done.** Every cell above corresponds to a screen in the live decisioning app and reads from the same artefacts the app consumes. Use section 6 to drill into any individual customer.